# Higgs Mechanism in the Standard Model

This notebook walks through every layer of the `higgs_mechanism` package applied to the
electroweak sector of the Standard Model: $G = \mathrm{SU}(2)_L \times U(1)_Y \to U(1)_{\mathrm{em}}$.

**Physical picture**

The SM Higgs doublet $H = (\phi^+, \phi^0)$ acquires a vacuum expectation value

$$\langle H \rangle = \frac{1}{\sqrt{2}}\begin{pmatrix}0\\v\end{pmatrix}, \qquad v \approx 246\ \text{GeV},$$

spontaneously breaking $\mathrm{SU}(2)_L \times U(1)_Y$ down to $U(1)_{\mathrm{em}}$.
Three generators are broken, producing three Goldstone bosons $G^\pm, G^0$ that are
absorbed by the $W^\pm$ and $Z$ bosons, making them massive. The photon remains massless.

**Expected tree-level spectrum**

| Field | Mass² |
|-------|-------|
| $W^\pm$ | $m_W^2 = g^2 v^2/4$ |
| $Z$ | $m_Z^2 = (g^2+g'^2)v^2/4$ |
| $A$ (photon) | $0$ |
| $h$ (Higgs boson) | $m_h^2 = 2\lambda v^2$ |
| $G^\pm, G^0$ | $0$ (Goldstones) |

In [1]:
from sympy import factor, simplify, pprint, latex, expand, sqrt, Rational
from IPython.display import display, Math

import higgs_mechanism as hm
from higgs_mechanism.gauge_group       import make_sm_gauge_group
from higgs_mechanism.scalar_sector     import make_sm_higgs_doublet
from higgs_mechanism.covariant_kinetic import build_sm_kinetic
from higgs_mechanism.scalar_potential  import (build_sm_potential, apply_tadpole_sm,
                                               build_sm_scalar_mass_matrix, identify_goldstones)
from higgs_mechanism.diagonalization   import (diagonalize_sm_gauge, verify_sm_gauge_masses,
                                               diagonalize_sm_scalar)
from higgs_mechanism.feynman_rules     import (build_sm_physical_lagrangian, extract_sm_vertices,
                                               build_sm_scalar_vertices)
from higgs_mechanism.validation        import run_sm_validation, print_validation_report

def show(expr, label=None):
    """Render a SymPy expression as LaTeX inline."""
    src = (f"{label} = " if label else "") + latex(expr)
    display(Math(src))

---
## Layer 0 — Gauge Group: $\mathrm{SU}(2)_L \times U(1)_Y$

We construct the electroweak gauge group with:
- **SU(2)_L**: gauge fields $W^1_\mu, W^2_\mu, W^3_\mu$, coupling $g$
- **U(1)_Y**: gauge field $B_\mu$, coupling $g'$

The covariant derivative acting on a doublet with hypercharge $Y = +\tfrac{1}{2}$ is:
$$D_\mu = \partial_\mu - i g W^a_\mu T^a - i g' Y B_\mu$$

In [4]:
group, gf = make_sm_gauge_group()

print(f"Gauge group : {group}")
print(f"Gauge fields: {group.all_gauge_fields}")
print()
for fac in group.factors:
    print(f"  {fac.name} ({fac.group_type}):  {fac.dim} generator(s),  coupling = {fac.g}")

print()
print("SU(2)_L generators T^a = σ^a/2 (fundamental representation):")
for a, T in enumerate(group.factors[0].generators_fundamental, start=1):
    print(f"  T^{a} =")
    show(T)

Gauge group : GaugeGroup([GaugeFactor(SU2_L, SU2, g=g), GaugeFactor(U1_Y, U1, g=g_prime)])
Gauge fields: [W1, W2, W3, B]

  SU2_L (SU2):  3 generator(s),  coupling = g
  U1_Y (U1):  1 generator(s),  coupling = g_prime

SU(2)_L generators T^a = σ^a/2 (fundamental representation):
  T^1 =


<IPython.core.display.Math object>

  T^2 =


<IPython.core.display.Math object>

  T^3 =


<IPython.core.display.Math object>

---
## Layer 1 — Scalar Sector: Higgs Doublet and VEV

The SM Higgs doublet is parameterised around its VEV as:

$$H = \begin{pmatrix}\phi^+\\\phi^0\end{pmatrix} = \begin{pmatrix}(G_1 + i G_2)/\sqrt{2}\\(v + h + i G^0)/\sqrt{2}\end{pmatrix}$$

where $h$ is the physical Higgs boson and $G^\pm, G^0$ are the would-be Goldstone bosons.

In [5]:
doublet, sf = make_sm_higgs_doublet(gf)

print(f"Multiplet  : {doublet}")
print(f"Components : {doublet.components}")
print(f"VEV values : {doublet.vev}")
print(f"Real d.o.f.: {2 * doublet.dim}  (4 real fields in total)")
print()
print("Field decompositions around the VEV:")
for field, expr in doublet.real_components.items():
    show(expr, label=str(field))

Multiplet  : ScalarMultiplet(SM_Higgs_doublet, dim=2)
Components : [\phi^+, \phi^0]
VEV values : [0, sqrt(2)*v/2]
Real d.o.f.: 4  (4 real fields in total)

Field decompositions around the VEV:


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

---
## Layer 2 — Gauge Boson Mass Matrix

Expanding the kinetic term $|D_\mu H|^2$ around the VEV yields a mass matrix for the
gauge bosons in the weak basis $\{W^1, W^2, W^3, B\}$:

$$\mathcal{L} \supset \frac{1}{2} A^a_\mu (M^2_{\text{gauge}})^{ab} A^{b\,\mu}$$

In [ ]:
L_mass, M2_gauge, vev_subs = build_sm_kinetic(gf, sf)

print("Gauge boson mass matrix M² in the [W¹, W², W³, B] basis:")
pprint(M2_gauge)

print()
g, gp, v = gf['g'], gf['g_prime'], sf['v']
print("Diagonal entries (mass-squared eigenvalues before rotation):")
for i, label in enumerate(['W¹', 'W²', 'W³', 'B']):
    show(factor(M2_gauge[i, i]), label=f"M²[{label},{label}]")

The off-diagonal $W^3$–$B$ mixing block is what produces the $Z$ boson and the
massless photon after diagonalisation.

---
## Layer 3 — Scalar Potential and Mass Matrix

The SM Higgs potential is:
$$V(H) = -\mu^2 H^\dagger H + \lambda (H^\dagger H)^2$$

The minimum condition (tadpole equation) $\partial V/\partial h\big|_{h=0} = 0$ gives
$\mu^2 = \lambda v^2$, and the scalar mass matrix is computed from the second derivatives.

In [ ]:
V, pot_params = build_sm_potential(sf)

print("Scalar potential V =  ", end="")
show(V)

tadpole_sol = apply_tadpole_sm(V, sf, pot_params)
print("Tadpole solution (minimum condition):")
for lhs, rhs in tadpole_sol.items():
    show(rhs, label=str(lhs))

M2_scalar, real_fields, V_real = build_sm_scalar_mass_matrix(V, sf, pot_params, tadpole_sol)

print()
print("Scalar mass matrix M²_S in the [h, G⁰, G₁, G₂] basis:")
pprint(M2_scalar)

In [ ]:
goldstones, physical, scalar_masses = identify_goldstones(M2_scalar, real_fields)

print(f"Physical Higgs fields : {physical}")
print(f"Goldstone bosons      : {goldstones}")
print()
print("Scalar mass-squared eigenvalues:")
for field, m2 in scalar_masses.items():
    show(factor(m2), label=f"m²({field})")

Three massless scalars ($G_1, G_2, G^0$) correspond to the three broken generators — confirmed by
Goldstone's theorem. The physical Higgs has $m_h^2 = 2\lambda v^2$.

---
## Layer 4 — Diagonalisation: Rotation to the Mass Basis

The $W^3$–$B$ system is rotated by the **Weinberg angle** $\theta_W$:

$$\begin{pmatrix}W^3_\mu \\ B_\mu\end{pmatrix} = \begin{pmatrix}\cos\theta_W & \sin\theta_W \\ -\sin\theta_W & \cos\theta_W\end{pmatrix}\begin{pmatrix}Z_\mu \\ A_\mu\end{pmatrix}$$

with $\tan\theta_W = g'/g$, giving the physical mass eigenstates $W^\pm, Z, A$.

In [ ]:
phys_fields, change_basis, gauge_mass_sq, weinberg = diagonalize_sm_gauge(gf, M2_gauge)

print("Physical gauge fields:", list(phys_fields.keys()))
print()
print("Weak-eigenstate → mass-eigenstate rotation:")
for k, expr in change_basis.items():
    show(expr, label=str(k))

print()
print("Weinberg angle relations:")
for k, expr in weinberg.items():
    show(factor(expr), label=str(k))

print()
print("Gauge boson masses squared:")
for name, m2 in gauge_mass_sq.items():
    show(factor(m2), label=f"m²({name})")

In [ ]:
# Verify the mass matrix is diagonal in the physical basis
M2_phys, checks = verify_sm_gauge_masses(M2_gauge, gf, change_basis)

print("Diagonality check on the rotated mass matrix:")
print(f"  Fully diagonal : {checks['diagonal']}")
print()
show(factor(checks['mW_sq']), label="m²(W)")
show(factor(checks['mZ_sq']), label="m²(Z)")
show(factor(checks['mA_sq']), label="m²(A)")

---
## Layer 5 — Feynman Rules

Expanding the physical Lagrangian in powers of the Higgs field $h$ extracts
the $HVV$ and scalar self-coupling vertices.
The SM predictions are:

$$g_{HWW} = \frac{g^2 v}{2}, \qquad g_{HZZ} = \frac{g^2 v}{2\cos^2\theta_W}, \qquad \frac{g_{HWW}}{g_{HZZ}} = \cos^2\theta_W$$

In [ ]:
L_int = build_sm_physical_lagrangian(gf, sf, weinberg)
vertices = extract_sm_vertices(L_int, sf, gf, weinberg)

print("Gauge-scalar vertices (computed vs. expected):")
print()
for name in ['HWW', 'HZZ', 'HHWW', 'HHZZ']:
    computed = factor(vertices[name])
    expected = factor(vertices[f'{name}_expected'])
    match = simplify(computed - expected) == 0
    status = "✓" if match else "✗"
    print(f"[{status}] g_{name}")
    show(computed, label="  computed ")
    show(expected, label="  expected ")
    print()

In [ ]:
scalar_vertices = build_sm_scalar_vertices(sf, pot_params, tadpole_sol)

print("Scalar self-coupling vertices:")
print()
for name in ['HHH', 'HHHH']:
    computed = factor(scalar_vertices[name])
    expected = factor(scalar_vertices[f'{name}_expected'])
    match = simplify(computed - expected) == 0
    status = "✓" if match else "✗"
    print(f"[{status}] g_{name}")
    show(computed, label="  computed ")
    show(expected, label="  expected ")
    print()

print("Higgs mass (consistency check):")
show(factor(scalar_vertices['mh_sq']),          label="  m²_h (computed)")
show(factor(scalar_vertices['mh_sq_expected']), label="  m²_h (expected)")

---
## Layer 6 — Validation Report

Automated consistency checks:
1. **Goldstone counting** — Goldstone's theorem: #(broken generators) = #(massless scalars)
2. **Weinberg relation** — $m_W^2/m_Z^2 = \cos^2\theta_W$
3. **Higgs mass** — $m_h^2 = 2\lambda v^2$
4. **Custodial symmetry** — $g_{HWW}/g_{HZZ} = \cos^2\theta_W$
5. **Gauge mass matrix rank** — exactly 1 massless gauge boson (photon)

In [ ]:
results = run_sm_validation(
    goldstones    = goldstones,
    scalar_masses = scalar_masses,
    gauge_mass_sq = gauge_mass_sq,
    vertices      = vertices,
    params        = pot_params,
    gauge_fields  = gf,
    weinberg      = weinberg,
)

print_validation_report(results)

---
## Summary — Physical Spectrum

All tree-level SM results reproduced symbolically:

| Field | Spin | Mass² | Origin |
|-------|------|-------|--------|
| $W^\pm$ | 1 | $g^2 v^2/4$ | 2 broken $\mathrm{SU}(2)_L$ generators |
| $Z$ | 1 | $(g^2+g'^2)v^2/4$ | 1 broken $\mathrm{SU}(2)_L \times U(1)_Y$ combination |
| $A$ (photon) | 1 | $0$ | $U(1)_{\mathrm{em}}$ remains unbroken |
| $h$ | 0 | $2\lambda v^2$ | Radial mode of $H$ |
| $G^\pm, G^0$ | 0 | $0$ | Goldstones — absorbed by $W^\pm, Z$ |

The Weinberg angle relates the couplings to the physical mixing:
$$\cos\theta_W = \frac{g}{\sqrt{g^2+g'^2}} = \frac{m_W}{m_Z}$$

In [ ]:
# Numerically illustrate with g=0.65, g'=0.36 (approximate SM values)
import sympy as sp

g_val, gp_val, v_val, lam_val = 0.6532, 0.3575, 246.0, 0.13
subs = {g: g_val, gp: gp_val, sf['v']: v_val, pot_params['lambda_']: lam_val}

print("Numerical mass spectrum (approximate SM values):")
print(f"  Input:  g = {g_val},  g' = {gp_val},  v = {v_val} GeV,  λ = {lam_val}")
print()

mW  = sp.sqrt(gauge_mass_sq['W^+'].subs(subs))
mZ  = sp.sqrt(gauge_mass_sq['Z'].subs(subs))
mh  = sp.sqrt(scalar_masses['h'].subs(subs))
thW = sp.acos(float(weinberg['cos_thW'].subs(subs))) * 180 / sp.pi

print(f"  m_W  = {float(mW):.2f} GeV   (PDG: 80.4 GeV)")
print(f"  m_Z  = {float(mZ):.2f} GeV   (PDG: 91.2 GeV)")
print(f"  m_h  = {float(mh):.2f} GeV   (PDG: 125  GeV, depends on λ)")
print(f"  θ_W  = {float(thW):.2f}°      (PDG: ~28.7°)")